# Twin experiment pipeline (realistic-noise edition) — what can satellite altimetry tell us about spatially variable bottom friction?

**One pipeline, from the solver to the decisions.** solver → noise model → independent data generator → gates → factors → information analysis → identification rows → noise sweep → pre-registered readings.

**What is new compared with the earlier notebook (which keeps the "3% of the window std" noise and stays valid for its own results).**
1. **Absolute noise in physical units**, identical for every U0, start type and data window (the earlier relative rule made σ depend on the tidal amplitude and on the transient, so cold-start and ramp rows were compared at different σ).
2. **Space-time correlated sea-level error** (tide-model/DAC-type residual) on top of white error; **velocity** noise is absolute and explicitly *hypothetical* (altimetry does not observe velocity).
3. **Exact GLS whitening** (the inversion knows the covariance: best case) or **white weights** (`WEIGHTING="white"`: the inversion ignores the correlation: the usual practice).
4. **Stage F: noise sweep** — asks whether h-only non-identification is noise-limited or structural.
5. A **noise-calibration table** prints the relative noise equivalent of every level, so the two notebooks can be compared.
6. Gates for the noise machinery itself (draw/whiten consistency) and an independent-generator gate judged against the smallest noise of the notebook.

**ASSUMPTION WARNING.** The noise levels are order-of-magnitude assumptions, not measurements of this project. The baseline `L2` (5 cm white + 3 cm correlated SLA; 3 cm/s velocity) must be replaced by values calibrated against tide gauges / product validation in the Strait before any statement about the real Strait.

| level | white SLA | correlated SLA | velocity | role |
|---|---|---|---|---|
| L0 | 1 cm | 0 | 0.5 cm/s | diagnostic floor (stage F) |
| L1 | 2 cm | 0 | 1 cm/s | optimistic |
| **L2** | 5 cm | 3 cm (25 km, 3 h) | 3 cm/s | **baseline (stages A-E)** |
| L3 | 10 cm | 6 cm | 6 cm/s | pessimistic |

**Stages (one `STAGE` word in cell 1; baseline level `NOISE_LEVEL`):**
| stage | content | cost (standard, estimate) |
|---|---|---|
| A | gates, noise calibration, transient decay (E1), linear information at all levels (E2) | 25-35 min |
| B | identification rows, early vs late window (E3) | ~2.5 h |
| C | same rows, smooth truth (E4) | ~1 h |
| D | cold start vs ramp at the optimiser level (E5) | ~2-2.5 h |
| E | independent data solver vs inversion solver (E6) | ~2 h |
| F | noise sweep (E7): levels L0, L1, L3 × U0 × {h, h+u+v} | ~2 h |
| ALL | everything (only with `MODE="quick"`) | ~40-50 min in quick |

**Pre-registration.** Four hashes (A-C, D, E, F) printed in the next cell; they include the noise model, the baseline level and the weighting, so results of different settings never mix. Thresholds are carried over unchanged from the earlier notebook, where several were set after seeing a preview: here they are a **re-test under another noise model, not blind predictions**.

**Honest scope.** Idealised channel; four-parameter zonal model; no PINN; no real-data statement; E2 is linearised at the true parameters.


In [1]:
import os, json, time, hashlib, math, glob
import numpy as np
from scipy.optimize import minimize
from scipy import stats
try:
    import jax, jax.numpy as jnp
    from jax import lax
    jax.config.update("jax_enable_x64", True)
    HAVE_JAX = True
except Exception as _e:
    HAVE_JAX = False
    print("WARNING: JAX not available ->", _e, "\n  GATE (JAX part) and all identification rows (E3-E7) are skipped; E2 falls back to NumPy finite differences.")

# ------------------------------------------------------------------------------------------------------------------
# The only things you normally change between sessions: MODE, STAGE (and, if you want another baseline, NOISE_LEVEL).
# ------------------------------------------------------------------------------------------------------------------
MODE = "standard"          # "quick" (pipeline check, ~30-40 min) | "standard"
STAGE = "C"              # "A" information | "B" identification rows | "C" smooth truth | "D" start type | "E" independent data solver | "F" noise sweep | "ALL" (quick only)
NOISE_LEVEL = "L2"         # baseline observation-noise level for stages A-E (levels are defined below). Stage F sweeps the other levels.
WEIGHTING = "gls"          # "gls": the inversion knows the noise covariance (best case) | "white": the inversion ignores the error correlation (what a practitioner usually does)
RUN = dict(GATE=True, E1=True, E2=STAGE in ("A", "ALL"), E3=STAGE in ("B", "ALL"), E4=STAGE in ("C", "ALL"),
           E5=STAGE in ("D", "ALL"), E6=STAGE in ("E", "ALL"), E7=STAGE in ("F", "ALL"))
RESUME = True
RAMP_DEFAULT = 3.0         # constant, never modified; E2 also evaluates ramp = 0 (cold start) explicitly
RAMP_HOURS = RAMP_DEFAULT  # current ramp; changed only through set_ramp()

# ------------------------------------------------------------------------------------------------------------------
# OBSERVATION-NOISE MODEL (absolute, physical units; independent of tidal amplitude, start type and data window)
# ------------------------------------------------------------------------------------------------------------------
# sea-level height h : e_h = white(sig_h_white) + correlated(sig_h_corr), the correlated part is a Gaussian random field in space
#                      (length L_space) times an exponential correlation in time (scale tau_time); it stands for tide-model/DAC residuals
#                      and slowly varying coastal-altimetry errors. Total sd = sqrt(white^2 + corr^2).
# velocity u, v      : white, sd = sig_vel (HYPOTHETICAL observation: satellite altimetry gives no velocity; ADCP-like in-situ scenario).
# ASSUMPTION WARNING : the numbers below are order-of-magnitude assumptions, NOT measured in this project. They must be replaced by values
#                      calibrated against tide gauges / validation of the X-TRACK/ALES+ product in the Strait (work package 'real data').
NOISE_LEVELS = {
    "L0": dict(sig_h_white=0.01, sig_h_corr=0.00, sig_vel=0.005, desc="diagnostic floor: 1 cm white SLA, 0.5 cm/s velocity (asks whether ANY noise level makes h-only identifiable)"),
    "L1": dict(sig_h_white=0.02, sig_h_corr=0.00, sig_vel=0.010, desc="optimistic: 2 cm white SLA, 1 cm/s velocity"),
    "L2": dict(sig_h_white=0.05, sig_h_corr=0.03, sig_vel=0.030, desc="baseline: 5 cm white + 3 cm correlated SLA (25 km, 3 h), 3 cm/s velocity"),
    "L3": dict(sig_h_white=0.10, sig_h_corr=0.06, sig_vel=0.060, desc="pessimistic: 10 cm white + 6 cm correlated SLA, 6 cm/s velocity"),
}
NOISE_CORR = dict(L_space_km=25.0, tau_time_h=3.0)
assert NOISE_LEVEL in NOISE_LEVELS and WEIGHTING in ("gls", "white")
NOISE = dict(level=NOISE_LEVEL)

def set_noise(level):
    assert level in NOISE_LEVELS
    NOISE["level"] = level

_NOISE_PRE = dict(levels=NOISE_LEVELS, corr=NOISE_CORR, baseline=NOISE_LEVEL, weighting=WEIGHTING,
                  text="absolute noise in physical units, independent of U0, start type and window; h: white + space-time correlated field; u,v: white (hypothetical observation); the inversion uses exact whitening ('gls') or white weights ('white') as set in WEIGHTING")

PREREG = dict(
    version="twin2-prereg-1",
    noise=_NOISE_PRE,
    question="Under an ABSOLUTE, partly correlated noise model (instead of 3% of the window std), what do linear information (E2) and multi-start identification (E3, E4) of the zonal 4-parameter model look like for the early (2.5-25 h) and a late data window, cold start and 3 h ramp?",
    window="data window = 54 snapshots, stride 25 steps (rich) or 4 snapshots (sparse4), first snapshot at step W0; W0=150 (2.5 h) is the early window",
    E1="periodicity defect d_k = rms(h(t)-h(t-P))/rms(h(t)) over successive blocks of one M2 period P (all cells, truth Cf, ramp 3 h); t_ss(U0) = end of the first block after which d_k < 0.05 for all later blocks (independent of the noise model)",
    E2="Fisher information of the zonal 4-parameter model at the true parameters (smooth truth), exact space-time noise covariance of the chosen level (whitened Jacobian); report median over layouts of sd(ln Cf_low), sd(ln Cf_high), sd_bnd_km = LX*sqrt(var(b0)+var(s)/12) for every level in E2_LEVELS; no absolute pass/fail",
    TH_A="h-only, rich, BASELINE level: median sd_bnd(cold start, W0=2.5 h) / sd_bnd(ramp 3 h, W0=2.5 h) <= 0.5 for every U0 in U0_E2 with U0 >= 0.5 -> 'transient carries information'; other levels are informational",
    TH_B="h-only, rich, ramp 3 h, BASELINE level: for every U0, min over windows W0 >= 24 h of sd_bnd(W0) / sd_bnd(W0=2.5 h) >= 0.67 -> 'no later window is >= 1.5x more informative'",
    carried_over="TH_A, TH_B and all thresholds are carried over UNCHANGED from the earlier notebook (3% relative noise), where they were set after seeing a NumPy preview; here they are a RE-TEST under another noise model, not blind predictions",
    E3="classical zonal multi-start rows (arm 'same', sharp truth, ramp 3 h), rich regime, variants h and huv, windows early (W0=150) and late (W0* from E1, clipped to [150, W0_CAP]); counts with Clopper-Pearson 95% intervals; 'late better than early' only if the intervals do not overlap",
    E4="same rows as E3 for U0=1 with a smooth truth Cf = cf_zonal(P_TRUE) (model family); reported; chi2_ref expected ~ 1",
    classification="DCHI2_CRIT 13.3, tolerances 25% (Cf) and 6.25 km (boundary), IDENTIFIED if identified >= ceil(0.9*valid), NOT_IDENTIFIED if <= floor(0.1*valid), else MARGINAL; chi2 is the (whitened) noise-normalised misfit",
    not_in_this_notebook="PINN, free-field lambda path, nuisance factors (wind, bathymetry, open boundary), real data",
)
PREREG_HASH = hashlib.sha256(json.dumps(PREREG, sort_keys=True).encode()).hexdigest()[:16]

PREREG_D = dict(
    version="twin2-D-prereg-1", noise=_NOISE_PRE,
    question="Under the absolute noise model, does the linear information advantage of the cold start translate into multi-start identification, and does a very late (~96 h) ramp-3 window help?",
    rows="classical zonal multi-start rows, arm 'same', sharp truth, h-only, rich regime: ramp3|early and cold|early for each U0 in U0_D; ramp3|late and cold|late (W0 = W0_D_LATE_H hours) for U0=1; baseline noise level",
    D1="cold start reproduces the earlier finding if, at U0=1, identified(cold,early) > identified(ramp3,early) with NON-overlapping 95% Clopper-Pearson intervals",
    D2="transient interpretation closed at the optimiser level only if D1 holds AND identified(cold,late) is not distinguishable from identified(ramp3,late) (overlapping intervals)",
    D3="'a ~96 h ramp-3 window helps h-only' only if identified(ramp3,late) > identified(ramp3,early) with non-overlapping intervals",
    honesty="in the earlier notebook (3% relative noise) D1 and D2 were supported and D3 not; those outcomes are known, so this stage is a re-test under another noise model, not a blind prediction",
)
PREREG_D_HASH = hashlib.sha256(json.dumps(PREREG_D, sort_keys=True).encode()).hexdigest()[:16]

PREREG_E = dict(
    version="twin2-E-prereg-1", noise=_NOISE_PRE,
    question="Under the absolute noise model, does generating the data with an independent solver (finer grid, RK4, different face friction; no 'inverse crime') change which rows are identified?",
    rows="classical zonal multi-start rows, ramp 3 h, sharp truth, rich regime, early window, variants h and huv, U0 in U0_E; arm 'same' and arm 'indep' with identical layouts and noise draws; baseline level",
    gate="generator volume balance (rel. error < 1e-6) and convergence (rms(G_r - G_2r) <= 0.25 x the SMALLEST noise sd of the notebook (level L0) for h, u, v at the generator refinement GEN_R) must pass before any row is interpreted",
    E1="'the independent generator changes identification' only if the 95% Clopper-Pearson intervals of identified(indep) and identified(same) do not overlap for that (U0, variant); otherwise 'no detectable change'",
    E2="model error (rms(inversion solver - generator)) in absolute units and relative to the baseline noise sd, and chi2_ref (mean, median, max) of the indep rows are reported; no pass/fail",
    honesty="in the earlier notebook the independent generator changed no row detectably; here the noise is absolute, so the (small) model error is compared with a different noise scale; not a blind prediction",
)
PREREG_E_HASH = hashlib.sha256(json.dumps(PREREG_E, sort_keys=True).encode()).hexdigest()[:16]

PREREG_F = dict(
    version="twin2-F-prereg-1", noise=_NOISE_PRE,
    question="Is the failure of h-only identification (ramp 3 h, early window) limited by observation noise, or does it persist when the noise is made very small? Does the advantage of adding velocity survive realistic noise?",
    rows="classical zonal multi-start rows, arm 'same', sharp truth, ramp 3 h, rich regime, early window, variants h and huv, U0 in U0_F, noise levels in F_LEVELS (the baseline level comes from the stage-B rows)",
    F1="'h-only non-identification is structural (not noise-limited within the tested range)' if identified(h, lowest level in F_LEVELS) <= floor(0.1*valid) for every U0 in U0_F; 'noise-limited' if identified >= ceil(0.9*valid) for some U0 at that level (sigma* = largest tested level that reaches IDENTIFIED); otherwise 'intermediate'",
    F2="'adding velocity helps at the baseline level' if for some U0 the lower Clopper-Pearson bound of identified(huv) exceeds the upper bound of identified(h) at the baseline level (rows from stage B)",
    F3="monotonic degradation (identified count non-increasing with sigma) is REPORTED per (U0, variant), not tested",
    honesty="no blind prediction is made; thresholds follow the same classification rule as stages B-E",
)
PREREG_F_HASH = hashlib.sha256(json.dumps(PREREG_F, sort_keys=True).encode()).hexdigest()[:16]

_MODES = {
    "quick":    dict(U0_E=[1.0], U0_D=[1.0], W0_D_LATE_H=30.0, N_STARTS=3, ZONAL_MAXITER=60, N_SEEDS=2, U0_E1=[0.5, 1.0], N_LONG_H=96, U0_E2=[1.0],
                     W0_E2_RAMP_H=[2.5, 24.0], W0_E2_COLD_H=[2.5, 24.0], N_LAYOUTS=2, U0_E3=[1.0], W0_CAP_H=40.0,
                     E2_LEVELS=["L1", "L2", "L3"], U0_F=[1.0], F_LEVELS=["L0", "L3"]),
    "standard": dict(U0_E=[0.5, 1.0], U0_D=[1.0, 0.5], W0_D_LATE_H=96.0, N_STARTS=6, ZONAL_MAXITER=120, N_SEEDS=10, U0_E1=[0.15, 0.3, 0.5, 1.0], U0_E2=[0.15, 0.5, 1.0],
                     W0_E2_RAMP_H=[2.5, 12.0, 24.0, 48.0, 96.0], W0_E2_COLD_H=[2.5, 24.0, 96.0], N_LAYOUTS=5, U0_E3=[0.5, 1.0], W0_CAP_H=96.0, N_LONG_H=240,
                     E2_LEVELS=["L0", "L1", "L2", "L3"], U0_F=[0.5, 1.0], F_LEVELS=["L0", "L1", "L3"]),
}
CFG = dict(N_STEPS=1500, T_STRIDE=25, WIN_LEN=1350, P_OBS=39, SEED=7, CF_SCALE=2.5e-3,
           TOL_BND_KM=6.25, TOL_CF_REL=0.25, DCHI2_CRIT=13.3, SS_THRESH=0.05, NOISE_LEVEL=NOISE_LEVEL, WEIGHTING=WEIGHTING)
CFG.update(_MODES[MODE])
REGIMES = {}       # filled by set_window()
GEN_R = 4          # refinement of the independent data generator: 96x48 cells, dt = 15 s, RK4

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_PATH = os.path.join(OUT_DIR, f"twin2_results_{MODE}_{NOISE_LEVEL}.json")   # cumulative file of this MODE and baseline level
SNAP_TAG = f"{MODE}_{NOISE_LEVEL}_stage{STAGE}"                                  # labels the archive copies written at the end of this session
_KEYS = ("gate", "noise_cal", "E1", "E2", "E3", "E4", "E5", "E6", "E7")
RESULTS = dict(mode=MODE, prereg=PREREG, prereg_hash=PREREG_HASH, prereg_D=PREREG_D, prereg_D_hash=PREREG_D_HASH, prereg_E=PREREG_E, prereg_E_hash=PREREG_E_HASH,
               prereg_F=PREREG_F, prereg_F_hash=PREREG_F_HASH, have_jax=HAVE_JAX, cfg=CFG, noise_levels=NOISE_LEVELS, noise_corr=NOISE_CORR,
               **{k: ({} if k != "noise_cal" else []) for k in _KEYS})
_resume_src = [p for p in [OUT_PATH] + sorted(glob.glob(f"/kaggle/input/**/twin2_results_{MODE}_{NOISE_LEVEL}.json", recursive=True)) if os.path.exists(p)]
if RESUME and _resume_src:
    _old = json.load(open(_resume_src[0]))
    if _old.get("prereg_hash") == PREREG_HASH and _old.get("mode") == MODE:
        for k in _KEYS: RESULTS[k] = _old.get(k, RESULTS[k])
        print("RESUMED from", _resume_src[0])
    else:
        print("WARNING: existing results file", _resume_src[0], "has a different pre-registration hash / mode -> NOT resumed.")

def _jsonable(o):
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))
def save_results():
    with open(OUT_PATH, "w") as f: json.dump(RESULTS, f, default=_jsonable)

print("MODE:", MODE, "| STAGE:", STAGE, "| baseline noise level:", NOISE_LEVEL, "-", NOISE_LEVELS[NOISE_LEVEL]["desc"], "| weighting:", WEIGHTING)
print("RUN:", RUN, "| output file:", OUT_PATH)
print("JAX:", HAVE_JAX, "| devices:", (jax.devices() if HAVE_JAX else "n/a"))
print("PRE-REGISTRATION HASHES  A-C:", PREREG_HASH, "| D:", PREREG_D_HASH, "| E:", PREREG_E_HASH, "| F:", PREREG_F_HASH)
print({k: v for k, v in CFG.items()})

MODE: standard | STAGE: C | baseline noise level: L2 - baseline: 5 cm white + 3 cm correlated SLA (25 km, 3 h), 3 cm/s velocity | weighting: gls
RUN: {'GATE': True, 'E1': True, 'E2': False, 'E3': False, 'E4': True, 'E5': False, 'E6': False, 'E7': False} | output file: /kaggle/working/twin2_results_standard_L2.json
JAX: True | devices: [CudaDevice(id=0), CudaDevice(id=1)]
PRE-REGISTRATION HASHES  A-C: fc780ac77f513334 | D: 086c7a70e942c1e9 | E: 609fe5ea67ae1b25 | F: 848edd06c215007a
{'N_STEPS': 1500, 'T_STRIDE': 25, 'WIN_LEN': 1350, 'P_OBS': 39, 'SEED': 7, 'CF_SCALE': 0.0025, 'TOL_BND_KM': 6.25, 'TOL_CF_REL': 0.25, 'DCHI2_CRIT': 13.3, 'SS_THRESH': 0.05, 'NOISE_LEVEL': 'L2', 'WEIGHTING': 'gls', 'U0_E': [0.5, 1.0], 'U0_D': [1.0, 0.5], 'W0_D_LATE_H': 96.0, 'N_STARTS': 6, 'ZONAL_MAXITER': 120, 'N_SEEDS': 10, 'U0_E1': [0.15, 0.3, 0.5, 1.0], 'U0_E2': [0.15, 0.5, 1.0], 'W0_E2_RAMP_H': [2.5, 12.0, 24.0, 48.0, 96.0], 'W0_E2_COLD_H': [2.5, 24.0, 96.0], 'N_LAYOUTS': 5, 'U0_E3': [0.5, 1.0], 'W0_CAP

## Solver, problem definition and observation-noise machinery

In [2]:
# ===== solver + problem definition (physics identical to the earlier notebook; new: observation-noise machinery) =====
# ---- core physics + problem definition (xp-generic: numpy for truth/tests, jax.numpy for inversion) ----
LX, LY = 150e3, 50e3
NX, NY = 24, 12
DX, DY = LX / NX, LY / NY
G, H0, F0, DT = 9.81, 80.0, 6.5e-5, 60.0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
RAMP_TR = RAMP_HOURS * 3600.0
def set_ramp(h):
    """change the smooth-start duration (0 = v3 cold start)."""
    global RAMP_TR, RAMP_HOURS
    RAMP_HOURS = h; RAMP_TR = h * 3600.0
def _ramp(xp, t):
    """smooth start of the boundary forcing: 0.5*(1-cos(pi t/TR)) for t < TR, then 1 (C1-continuous). RAMP_HOURS = 0 -> v3 behaviour."""
    if RAMP_TR <= 0: return 1.0
    return xp.where(t < RAMP_TR, 0.5 * (1.0 - xp.cos(xp.pi * t / RAMP_TR)), 1.0)
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
B0_TRUE, S_TRUE = 0.5, 0.1          # boundary x/LX = B0 + S*(y/LY - 0.5)  (identical to the phase-0 twin)
XC = (np.arange(NX) + 0.5) * DX / LX     # normalized cell-centre coordinates
YC = (np.arange(NY) + 0.5) * DY / LY

def swe_step(xp, state, t, Cf, U0):
    eta, u, v = state
    vp = xp.pad(v, ((1, 1), (0, 0)), mode="edge"); vc = 0.5 * (vp[:, :-1] + vp[:, 1:]); v_at_u = 0.5 * (vc[:-1, :] + vc[1:, :])
    uc = 0.5 * (u[:-1, :] + u[1:, :]); up = xp.pad(uc, ((0, 0), (1, 1)), mode="edge"); u_at_v = 0.5 * (up[:, :-1] + up[:, 1:])
    Cf_u = Cf[np.clip(np.arange(NX + 1), 0, NX - 1), :]
    Cf_v = Cf[:, np.clip(np.arange(NY + 1), 0, NY - 1)]
    su = xp.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = xp.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    de_dx = (eta[1:, :] - eta[:-1, :]) / DX; de_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fu = Cf_u / H0 * u * su; fv = Cf_v / H0 * v * sv
    un = u[1:-1, :] + DT * (-G * de_dx + F0 * v_at_u[1:-1, :] - fu[1:-1, :])
    u_new = xp.concatenate([xp.full((1, NY), U0 * _ramp(xp, t) * xp.cos(OMEGA * t)), un, xp.zeros((1, NY))], axis=0)
    vn = v[:, 1:-1] + DT * (-G * de_dy - F0 * u_at_v[:, 1:-1] - fv[:, 1:-1])
    v_new = xp.concatenate([xp.zeros((NX, 1)), vn, xp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def run_obs_np(Cf, U0, n_steps, ox, oy):
    """numpy reference run -> (h,u,v) at observed cells for every step: 3 arrays (n_steps, P)."""
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H, U, V = [], [], []
    for n in range(n_steps):
        s = swe_step(np, s, n * DT, Cf, U0)
        e, u, v = s
        H.append(e[ox, oy]); U.append((0.5 * (u[:-1] + u[1:]))[ox, oy]); V.append((0.5 * (v[:, :-1] + v[:, 1:]))[ox, oy])
    return np.array(H), np.array(U), np.array(V)

def cf_zonal(xp, p, w_cells=0.5):
    """p = [ln Cf_low, ln Cf_high, b0, s]; smooth two-zone field on the cell grid (width = w_cells cells)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    arg = (X - (p[2] + p[3] * (Y - 0.5))) / (w_cells * DX / LX)
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(p[0]), xp.exp(p[1])
    return lo + (hi - lo) * sig

def cf_true_field():
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    return np.where(X < B0_TRUE + S_TRUE * (Y - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE)


# ---- K-zone model: K strips along x separated by K-1 parallel (common-slope) smooth boundaries ----
def cf_zonal_K(xp, theta, K, w_cells=0.5):
    """theta = [ln c_1..ln c_K, b_1..b_{K-1}, s]  (K=1: [ln c_1]).  Boundary k: x/LX = b_k + s*(y/LY-0.5)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    f = xp.exp(theta[0]) * xp.ones((NX, NY))
    for k in range(K - 1):
        arg = (X - (theta[K + k] + theta[2 * K - 1] * (Y - 0.5))) / (w_cells * DX / LX)
        f = f + (xp.exp(theta[k + 1]) - xp.exp(theta[k])) / (1.0 + xp.exp(-arg))
    return f



# =====================================================================================================================
# OBSERVATION-NOISE MACHINERY (NumPy; independent of JAX)
# =====================================================================================================================
# Covariance of the sea-level noise on a (time T x cells P) window:  C = sig_w^2 I + sig_c^2 (Kt (x) Ks)
#   Ks = Gaussian kernel on cell-centre distances (length L_space), Kt = exp(-|dt|/tau) on the actual snapshot times.
# Eigen-decompositions Ks = Us diag(ls) Us^T and Kt = Ut diag(lt) Ut^T diagonalise C exactly, so
#   draw:    e = sig_w z + sig_c (Ut sqrt(lt)) Z (Us sqrt(ls))^T     (z, Z standard normal T x P)
#   whiten:  r_w = (Ut^T r Us) / sqrt(sig_w^2 + sig_c^2 lt_i ls_j)   ->  E||r_w||^2 / (T P) = 1 for r = e  (exact GLS misfit)
# weighting 'white' instead divides by the total sd sqrt(sig_w^2 + sig_c^2) and ignores the correlation (mis-specified covariance).
def noise_spec(level=None):
    return NOISE_LEVELS[NOISE["level"] if level is None else level]

def _kernels(ox, oy, t_idx):
    xs = (np.asarray(ox, float) + 0.5) * DX; ys = (np.asarray(oy, float) + 0.5) * DY
    d2 = (xs[:, None] - xs[None, :]) ** 2 + (ys[:, None] - ys[None, :]) ** 2
    Ks = np.exp(-d2 / (2.0 * (NOISE_CORR["L_space_km"] * 1e3) ** 2))
    tt = np.asarray(t_idx, float) * DT / 3600.0
    Kt = np.exp(-np.abs(tt[:, None] - tt[None, :]) / NOISE_CORR["tau_time_h"])
    return Ks, Kt

_NZ_CACHE = {}
def noise_ctx(ox, oy, t_idx, level=None, weighting=None):
    """everything needed to draw and to whiten the noise of one window/layout at one level."""
    sp = noise_spec(level); weighting = WEIGHTING if weighting is None else weighting
    key = (tuple(int(a) for a in ox), tuple(int(a) for a in oy), tuple(int(a) for a in t_idx))
    if key not in _NZ_CACHE:
        Ks, Kt = _kernels(ox, oy, t_idx)
        ls, Us = np.linalg.eigh(Ks); lt, Ut = np.linalg.eigh(Kt)
        _NZ_CACHE[key] = (Us, np.clip(ls, 0.0, None), Ut, np.clip(lt, 0.0, None))
    Us, ls, Ut, lt = _NZ_CACHE[key]
    sw, sc = sp["sig_h_white"], sp["sig_h_corr"]
    var = sw ** 2 + sc ** 2 * (lt[:, None] * ls[None, :])
    inv_gls = 1.0 / np.sqrt(var)
    tot = math.sqrt(sw ** 2 + sc ** 2)
    ctx = dict(Us=Us, ls=ls, Ut=Ut, lt=lt, sig_h_white=sw, sig_h_corr=sc, sig_vel=sp["sig_vel"], sig_h_total=tot, inv_gls=inv_gls,
               weighting=weighting, level=(NOISE["level"] if level is None else level))
    if weighting == "gls":
        ctx.update(Wt=Ut, Ws=Us, inv=inv_gls)
    else:
        ctx.update(Wt=None, Ws=None, inv=np.full_like(inv_gls, 1.0 / tot))
    return ctx

def draw_noise(ctx, T, P, noise_seed, variant):
    """common random numbers: the same standard-normal draws are scaled by the sd of every level."""
    rng_w = np.random.default_rng(11 + noise_seed); rng_c = np.random.default_rng(5011 + noise_seed)
    e = dict(h=ctx["sig_h_white"] * rng_w.normal(size=(T, P)))
    Zc = rng_c.normal(size=(T, P))
    if ctx["sig_h_corr"] > 0:
        e["h"] = e["h"] + ctx["sig_h_corr"] * ((ctx["Ut"] * np.sqrt(ctx["lt"])) @ Zc @ (ctx["Us"] * np.sqrt(ctx["ls"])).T)
    if variant == "huv":
        e["u"] = ctx["sig_vel"] * rng_w.normal(size=(T, P)); e["v"] = ctx["sig_vel"] * rng_w.normal(size=(T, P))
    return e

def make_obs(clean, ox, oy, t_idx, variant, noise_seed, level=None):
    ctx = noise_ctx(ox, oy, t_idx, level)
    T, P = clean["h"].shape
    e = draw_noise(ctx, T, P, noise_seed, variant)
    return ctx, {k: clean[k] + e[k] for k in e}

def make_chi2(fwd, t_idx, obs, ctx, variant, xp=None):
    """noise-normalised misfit of a forward model fwd(Cf)->(H,U,V) (series at the observed cells): mean of squared whitened residuals.
    h: exact GLS whitening (or white weights), u,v: white. xp = jax.numpy (default) or numpy (tests)."""
    xp = jnp if xp is None else xp
    tj = xp.asarray(np.asarray(t_idx)); inv = xp.asarray(ctx["inv"]); sv = ctx["sig_vel"]
    if ctx["Wt"] is not None: Wt = xp.asarray(ctx["Wt"]); Ws = xp.asarray(ctx["Ws"])
    def chi2(Cf):
        H, U, V = fwd(Cf)
        r = H[tj] - obs["h"]
        if ctx["Wt"] is not None: r = (Wt.T @ r) @ Ws
        c = xp.mean((r * inv) ** 2)
        if variant == "huv":
            c = (c + xp.mean(((U[tj] - obs["u"]) / sv) ** 2) + xp.mean(((V[tj] - obs["v"]) / sv) ** 2)) / 3.0
        return c
    return chi2


CF_T = cf_true_field()
BOUNDS = [(np.log(5e-4), np.log(2e-2))] * 2 + [(0.1, 0.9), (-0.5, 0.5)]
P_TRUE = np.array([np.log(CF_LOW_TRUE), np.log(CF_HIGH_TRUE), B0_TRUE, S_TRUE])
if HAVE_JAX:
    # ---- JAX forward model returning only the observed cells (keeps memory tiny) ----
    def make_forward(U0, ox, oy):
        oxj, oyj = jnp.array(ox), jnp.array(oy)
        def fwd(Cf):
            def body(state, n):
                new = swe_step(jnp, state, n * DT, Cf, U0)
                e, u, v = new
                uc = 0.5 * (u[:-1, :] + u[1:, :]); vc = 0.5 * (v[:, :-1] + v[:, 1:])
                return new, (e[oxj, oyj], uc[oxj, oyj], vc[oxj, oyj])
            init = (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))
            _, ys = lax.scan(body, init, jnp.arange(CFG["N_STEPS"]))
            return ys
        return fwd

## Tools: data windows, transient analysis, linear information (GLS-whitened), independent generator, classical rows

In [3]:
# ===================== tools: data windows, transient analysis, linear information, independent generator, classical rows =====================
def regime_idx(W0, regime):
    """snapshot steps of a data window whose first snapshot is at step W0 (W0=150 == v3f window)."""
    if regime == "rich":    return np.arange(W0, W0 + CFG["WIN_LEN"], CFG["T_STRIDE"])
    if regime == "sparse4": return W0 + np.array([225, 599, 974, 1349])
    raise ValueError(regime)

def set_window(W0):
    """set N_STEPS and REGIMES for a data window starting at step W0 (used by the v3f problem builders, which read these globals)."""
    CFG["N_STEPS"] = int(W0 + CFG["WIN_LEN"])
    REGIMES["rich"] = regime_idx(W0, "rich"); REGIMES["sparse4"] = regime_idx(W0, "sparse4")
set_window(150)

def cp_interval(k, n, alpha=0.05):
    """Clopper-Pearson interval for k successes out of n."""
    lo = 0.0 if k == 0 else stats.beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else stats.beta.ppf(1 - alpha / 2, k + 1, n - k)
    return float(lo), float(hi)

# ---------------- E1: transient decay (NumPy only) ----------------
P_STEPS = 2 * np.pi / OMEGA / DT           # M2 period in steps (non-integer)

def run_full_np(Cf, U0, n):
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H = np.empty((n, NX, NY))
    for k in range(n):
        s = swe_step(np, s, k * DT, Cf, U0); H[k] = s[0]
    return H

def periodicity_defect(H):
    """list of (block end time [h], d_k) for blocks k = 1, 2, ... of one M2 period (block k compared with the same phase one period earlier)."""
    out, k = [], 1
    while (k + 1) * P_STEPS <= len(H):
        idx = np.arange(int(k * P_STEPS), int((k + 1) * P_STEPS))
        prev = idx - P_STEPS; lo = np.floor(prev).astype(int); fr = prev - lo
        hp = H[lo] * (1 - fr)[:, None, None] + H[lo + 1] * fr[:, None, None]
        d = np.sqrt(np.mean((H[idx] - hp) ** 2)) / np.sqrt(np.mean(H[idx] ** 2))
        out.append(((k + 1) * P_STEPS * DT / 3600.0, float(d))); k += 1
    return out

def steady_time(blocks, thr):
    """end time [h] of the first block after which every block has d < thr; None if never."""
    for i in range(len(blocks)):
        if all(b[1] < thr for b in blocks[i:]): return blocks[i][0]
    return None

def residual_period(H, cell=None, n_harm=4):
    """dominant period [h] and relative std of the residual of one mid-channel cell over t in [P, 3P] after fitting const + trend + M2 and its overtones."""
    cell = (NX // 2, NY // 2) if cell is None else cell
    a, b = int(P_STEPS), min(int(3 * P_STEPS), len(H))
    t = np.arange(a, b) * DT; y = H[a:b, cell[0], cell[1]]
    cols = [np.ones_like(t), t - t.mean()]
    for m in range(1, n_harm + 1): cols += [np.cos(m * OMEGA * t), np.sin(m * OMEGA * t)]
    A = np.stack(cols, 1); coef = np.linalg.lstsq(A, y, rcond=None)[0]; r = y - A @ coef
    rw = r * np.hanning(len(r)); nfft = 16 * len(r)
    sp = np.abs(np.fft.rfft(rw, nfft)); f = np.fft.rfftfreq(nfft, DT)
    i = int(np.argmax(sp[1:])) + 1
    return float(1.0 / f[i] / 3600.0), float(np.std(r) / np.std(y))

# ---------------- E2: linearised information ----------------
def layout_cells(layout):
    rng = np.random.default_rng(CFG["SEED"] + layout)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    return np.unravel_index(cells, (NX, NY))

def jac_fd(U0, ox, oy, n, h=1e-4):
    """NumPy central finite differences of the observed (h,u,v) series w.r.t. the 4 zonal parameters at P_TRUE (smooth truth)."""
    base = run_obs_np(cf_zonal(np, P_TRUE), U0, n, ox, oy); cols = []
    for i in range(4):
        pp = P_TRUE.copy(); pp[i] += h; pm = P_TRUE.copy(); pm[i] -= h
        a = run_obs_np(cf_zonal(np, pp), U0, n, ox, oy); b = run_obs_np(cf_zonal(np, pm), U0, n, ox, oy)
        cols.append([(x - y) / (2 * h) for x, y in zip(a, b)])
    return list(base), [np.stack([cols[i][v] for i in range(4)], -1) for v in range(3)]

if HAVE_JAX:
    def jac_jax(U0, ox, oy, n):
        old = CFG["N_STEPS"]; CFG["N_STEPS"] = n
        try:
            fwd = make_forward(U0, ox, oy)
            g = lambda p: fwd(cf_zonal(jnp, p))
            both = jax.jit(lambda p: (g(p), jax.jacfwd(g)(p)))
            base, J = both(jnp.array(P_TRUE))
        finally:
            CFG["N_STEPS"] = old
        return [np.asarray(b) for b in base], [np.asarray(j) for j in J]

def fisher_metrics(J, t_idx, variant, ox, oy, level):
    """linearised uncertainty of (ln Cf_low, ln Cf_high, boundary offset b0, tilt s) for the window t_idx at one noise level, with the exact
    space-time noise covariance of that level (GLS-whitened Jacobian). h-only: all P_OBS cells; huv: first P_OBS//3 cells, h+u+v (u, v white)."""
    Pu = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    ctx = noise_ctx(np.asarray(ox)[:Pu], np.asarray(oy)[:Pu], t_idx, level, weighting="gls")
    Jh = J[0][t_idx][:, :Pu, :]
    W = np.einsum("ta,tpk,pb->abk", ctx["Ut"], Jh, ctx["Us"]) * ctx["inv_gls"][:, :, None]
    rows = [W.reshape(-1, 4)]
    if variant == "huv":
        for vi in (1, 2): rows.append(J[vi][t_idx][:, :Pu, :].reshape(-1, 4) / ctx["sig_vel"])
    Jt = np.vstack(rows); F = Jt.T @ Jt
    sv = np.linalg.svd(Jt, compute_uv=False)
    C = np.linalg.pinv(F); sg = np.sqrt(np.clip(np.diag(C), 0, None))
    return dict(sd_lnClow=float(sg[0]), sd_lnChigh=float(sg[1]), sd_b0_km=float(sg[2] * LX / 1e3), sd_tilt_km=float(sg[3] * LX / 1e3),
                sd_bnd_km=float(LX / 1e3 * math.sqrt(C[2, 2] + C[3, 3] / 12.0)), sv=[float(x) for x in sv], cond_F=float(np.linalg.cond(F)))

# ---------------- independent data generator (NumPy; same equations, different numerics) ----------------
# ======================= independent generator =======================
# Same continuous equations as the inversion solver (no advection, no viscosity, quadratic friction, f-plane, same ramp),
# DIFFERENT numerics: r-times finer grid, classical RK4 in time, truth front area-averaged on the fine cells, face friction = mean of the two neighbouring cells,
# boundary forcing evaluated at the true time. Truth Cf is evaluated at the fine cell centres (sharp step on the fine grid).
def _grid(r):
    nx, ny = NX * r, NY * r
    return nx, ny, LX / nx, LY / ny, (np.arange(nx) + 0.5) / nx, (np.arange(ny) + 0.5) / ny

def _rhs(eta, u, v, t, Cf_u, Cf_v, U0, dx, dy):
    u = u.copy(); v = v.copy()
    u[0, :] = U0 * float(_ramp(np, t)) * np.cos(OMEGA * t); u[-1, :] = 0.0
    v[:, 0] = 0.0; v[:, -1] = 0.0
    v_c = 0.5 * (v[:, :-1] + v[:, 1:]); u_c = 0.5 * (u[:-1, :] + u[1:, :])
    v_at_u = np.zeros_like(u); v_at_u[1:-1, :] = 0.5 * (v_c[:-1, :] + v_c[1:, :])
    u_at_v = np.zeros_like(v); u_at_v[:, 1:-1] = 0.5 * (u_c[:, :-1] + u_c[:, 1:])
    su = np.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = np.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    du = np.zeros_like(u); dv = np.zeros_like(v)
    du[1:-1, :] = -G * (eta[1:, :] - eta[:-1, :]) / dx + F0 * v_at_u[1:-1, :] - Cf_u[1:-1, :] / H0 * u[1:-1, :] * su[1:-1, :]
    dv[:, 1:-1] = -G * (eta[:, 1:] - eta[:, :-1]) / dy - F0 * u_at_v[:, 1:-1] - Cf_v[:, 1:-1] / H0 * v[:, 1:-1] * sv[:, 1:-1]
    deta = -H0 * ((u[1:, :] - u[:-1, :]) / dx + (v[:, 1:] - v[:, :-1]) / dy)
    return deta, du, dv

def _block_mean(a, r):
    return a.reshape(NX, r, NY, r).mean(axis=(1, 3))

def run_generator(cf_fn, U0, r=2, n_steps=None, rec_volume=False):
    """(h, u, v) at COARSE cell centres, shape (n_steps, NX, NY); entry n = state at time (n+1)*DT, exactly like run_obs_np
    and the JAX forward model. Fine fields are block-averaged r x r onto the coarse cells."""
    n_steps = CFG["N_STEPS"] if n_steps is None else n_steps
    nx, ny, dx, dy, xc, yc = _grid(r)
    X, Y = np.meshgrid(xc, yc, indexing="ij")
    Cf = cf_fn(X, Y)
    Cf_u = np.zeros((nx + 1, ny)); Cf_u[1:-1, :] = 0.5 * (Cf[:-1, :] + Cf[1:, :])
    Cf_v = np.zeros((nx, ny + 1)); Cf_v[:, 1:-1] = 0.5 * (Cf[:, :-1] + Cf[:, 1:])
    eta = np.zeros((nx, ny)); u = np.zeros((nx + 1, ny)); v = np.zeros((nx, ny + 1))
    dt = DT / r
    Hh = np.empty((n_steps, NX, NY)); Uu = np.empty_like(Hh); Vv = np.empty_like(Hh); vol = np.empty(n_steps)
    f = lambda e, uu, vv, t: _rhs(e, uu, vv, t, Cf_u, Cf_v, U0, dx, dy)
    for k in range(1, n_steps * r + 1):
        t = (k - 1) * dt
        k1 = f(eta, u, v, t)
        k2 = f(eta + 0.5 * dt * k1[0], u + 0.5 * dt * k1[1], v + 0.5 * dt * k1[2], t + 0.5 * dt)
        k3 = f(eta + 0.5 * dt * k2[0], u + 0.5 * dt * k2[1], v + 0.5 * dt * k2[2], t + 0.5 * dt)
        k4 = f(eta + dt * k3[0], u + dt * k3[1], v + dt * k3[2], t + dt)
        eta = eta + dt / 6.0 * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0])
        u = u + dt / 6.0 * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1])
        v = v + dt / 6.0 * (k1[2] + 2 * k2[2] + 2 * k3[2] + k4[2])
        if k % r == 0:
            n = k // r - 1
            u[0, :] = U0 * float(_ramp(np, k * dt)) * np.cos(OMEGA * k * dt); u[-1, :] = 0.0   # boundary faces hold the imposed value at the new time
            Hh[n] = _block_mean(eta, r)
            Uu[n] = _block_mean(0.5 * (u[:-1, :] + u[1:, :]), r)
            Vv[n] = _block_mean(0.5 * (v[:, :-1] + v[:, 1:]), r)
            vol[n] = eta.sum() * dx * dy
    return (Hh, Uu, Vv, vol) if rec_volume else (Hh, Uu, Vv)

_B = lambda X, Y: B0_TRUE + S_TRUE * (Y - 0.5)
TRUTH_FN_SHARP = {      # point-wise definitions (the coarse 'same' arm evaluates these at coarse cell centres = v3 truths)
    "standard":   lambda X, Y: np.where(X < _B(X, Y), CF_LOW_TRUE, CF_HIGH_TRUE),
    "gradient":   lambda X, Y: CF_LOW_TRUE + (CF_HIGH_TRUE - CF_LOW_TRUE) * X,
    "uniform":    lambda X, Y: np.full_like(X, 2.5e-3),
}
def _antialias(fn, s=8):
    """area-average of fn over each generator cell (s x s sub-samples): removes the staircase of a sharp tilted front, which otherwise
    makes the (tiny) transverse velocity v converge only at first order with refinement."""
    def g(X, Y):
        nx, ny = X.shape; off = (np.arange(s) + 0.5) / s - 0.5; acc = np.zeros_like(X)
        for a in off:
            for b in off: acc += fn(X + a / nx, Y + b / ny)
        return acc / (s * s)
    return g
TRUTH_FN = {k: _antialias(f) for k, f in TRUTH_FN_SHARP.items()}
_GEN_CACHE = {}
def gen_fields(truth, U0, r=None):
    r = GEN_R if r is None else r
    key = (truth, float(U0), int(r), CFG["N_STEPS"], RAMP_HOURS)
    if key not in _GEN_CACHE: _GEN_CACHE[key] = run_generator(TRUTH_FN[truth], U0, r)
    return _GEN_CACHE[key]


# ---------------- classical rows (arms 'same' and 'indep') ----------------
ARM = "same"
def _layout_for_seed(P_use, noise_seed):
    rng = np.random.default_rng(CFG["SEED"] + noise_seed)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    return np.unravel_index(cells[:P_use], (NX, NY))

def build_problem_from_clean(U0, regime, variant, P_use, noise_seed, clean_fn, level=None):
    """common part of both arms: layout (same seeds), noise (common random numbers), forward model and misfit. clean_fn(ox, oy, t_idx) -> dict(h,u,v) of shape (T, P)."""
    ox, oy = _layout_for_seed(P_use, noise_seed)
    t_idx = REGIMES[regime]
    clean = clean_fn(ox, oy, t_idx)
    ctx, obs = make_obs(clean, ox, oy, t_idx, variant, noise_seed, level)
    fwd = make_forward(U0, ox, oy)
    return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, ctx=ctx, fwd=fwd,
                chi2=make_chi2(fwd, t_idx, obs, ctx, variant), n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

if HAVE_JAX:
    def build_problem(U0, regime, variant, P_use, noise_seed=0, cf_truth=None, level=None):
        if ARM == "same":      # truth from the NumPy loop of the SAME scheme as the inversion solver ('inverse crime')
            cf = CF_T if cf_truth is None else cf_truth
            def clean_fn(ox, oy, t_idx):
                Hc, Uc, Vc = run_obs_np(cf, U0, CFG["N_STEPS"], ox, oy)
                return dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
        else:                  # clean data from the independent generator
            if cf_truth is not None: raise ValueError("arm 'indep': only the standard sharp truth is wired")
            def clean_fn(ox, oy, t_idx):
                Hf, Uf, Vf = gen_fields("standard", U0)
                return dict(h=Hf[t_idx][:, ox, oy], u=Uf[t_idx][:, ox, oy], v=Vf[t_idx][:, ox, oy])
        return build_problem_from_clean(U0, regime, variant, P_use, noise_seed, clean_fn, level)

    def fit_zonal(pb, p0):
        vgf = jax.jit(jax.value_and_grad(lambda p: pb["chi2"](cf_zonal(jnp, p))))
        def f(p):
            v, g = vgf(jnp.array(p)); return float(v), np.asarray(g, dtype=np.float64)
        res = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=BOUNDS, options=dict(maxiter=CFG["ZONAL_MAXITER"], ftol=1e-12, gtol=1e-8))
        p = res.x
        off = float((p[2] - B0_TRUE) * LX / 1e3); tilt = float((p[3] - S_TRUE) * LX / 1e3)
        return dict(cf_low=float(np.exp(p[0])), cf_high=float(np.exp(p[1])), b0=float(p[2]), s=float(p[3]), chi2=float(res.fun), nit=int(res.nit),
                    bnd_km=float(np.mean(np.abs((p[2] - B0_TRUE) + (p[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3),
                    off_km=off, tilt_km=tilt, edge_km=float(max(abs(off - tilt / 2), abs(off + tilt / 2))))

    def zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=None, level=None):
        pbz = build_problem(U0, regime, variant, P_use, noise_seed=ns, cf_truth=cf_truth, level=level)
        chi2_ref = float(pbz["chi2"](cf_zonal(jnp, jnp.array(P_TRUE))))
        sols = []
        for k in range(CFG["N_STARTS"]):
            r = np.random.default_rng(100 + k + 17 * ns)
            p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
            sols.append(fit_zonal(pbz, p0))
        return dict(noise_seed=ns, chi2_ref=chi2_ref, n_obs=pbz["n_obs"], sols=sols, level=pbz["ctx"]["level"], weighting=pbz["ctx"]["weighting"])

def classify_seed(sd, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    n = sd["n_obs"]; sols = sd["sols"]; best = min(s["chi2"] for s in sols)
    if best > sd["chi2_ref"] + 2 * np.sqrt(2 / n): return "optimiser_failed"
    equiv = [s for s in sols if n * (s["chi2"] - best) <= CFG["DCHI2_CRIT"]]
    key = "edge_km" if use_edge else "bnd_km"
    ok = all(s[key] <= tol_bnd and abs(s["cf_low"] / CF_LOW_TRUE - 1) <= tol_cf and abs(s["cf_high"] / CF_HIGH_TRUE - 1) <= tol_cf for s in equiv)
    return "identified" if ok else "not_identified"

def misspec_flag(sd): return sd["chi2_ref"] > 1 + 5 * np.sqrt(2 / sd["n_obs"])

def classify_row(per_seed, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    labs = [classify_seed(sd, tol_cf, tol_bnd, use_edge) for sd in per_seed]
    ni, nn, nf = labs.count("identified"), labs.count("not_identified"), labs.count("optimiser_failed")
    valid = ni + nn
    if valid < 0.8 * len(per_seed) or valid == 0: cls = "INVALID"
    elif ni >= math.ceil(0.9 * valid): cls = "IDENTIFIED"
    elif ni <= math.floor(0.1 * valid): cls = "NOT_IDENTIFIED"
    else: cls = "MARGINAL"
    return dict(identified=ni, not_identified=nn, optimiser_failed=nf, n_seeds=len(per_seed), cls=cls)

def late_W0(U0):
    """first window start (steps, multiple of the stride) at or after the steady time t_ss(U0) from E1, clipped to [150, W0_CAP]."""
    cap = int(CFG["W0_CAP_H"] * 3600 / DT)
    e1 = RESULTS["E1"].get(str(U0), {})
    t_ss = e1.get("t_ss_h")
    flag = "ok"
    if t_ss is None:
        W0 = cap; flag = "steady state NOT reached in E1 run -> cap"
    else:
        W0 = int(math.ceil(t_ss * 3600 / DT / CFG["T_STRIDE"]) * CFG["T_STRIDE"])
        if W0 > cap: W0 = cap; flag = "t_ss beyond cap -> cap"
    return max(150, W0), flag

def run_row(store, tag, U0, variant, n_seeds, W0, cf_truth=None, regime="rich", arm="same", level=None):
    """run (or resume) one classical row for the window starting at step W0 at noise level `level` (default: baseline); results stored in store[tag]."""
    global ARM
    lvl = NOISE_LEVEL if level is None else level
    ARM = arm; set_window(W0); set_noise(lvl)
    P_use = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    rows = store.setdefault(tag, dict(U0=U0, variant=variant, regime=regime, W0=int(W0), arm=arm, level=lvl, weighting=WEIGHTING, seeds=[]))["seeds"]
    t0 = time.time()
    for ns in range(len(rows), n_seeds):
        rows.append(zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=cf_truth, level=lvl)); save_results()
    c = classify_row(rows); lo, hi = cp_interval(c["identified"], c["n_seeds"])
    mc = float(np.mean([s["chi2_ref"] for s in rows]))
    print(f"[row] {tag:46s} W0={W0*DT/3600:6.1f} h  noise={lvl}  identified {c['identified']:2d}/{c['n_seeds']}  CP95=[{lo:.2f},{hi:.2f}]  -> {c['cls']:15s} chi2_ref(mean)={mc:5.2f}  flagged={sum(misspec_flag(s) for s in rows)}  [{time.time()-t0:.0f}s]")
    set_window(150); set_noise(NOISE_LEVEL); ARM = "same"
    return c

## GATE: forward model, Jacobian, noise-model self-test

In [4]:
# ===================== GATE =====================

# ---- noise-model self-test (NumPy only): the draw and the whitening must be exactly consistent ----
def noise_selftest():
    ok = {}; rng = np.random.default_rng(0)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False); ox, oy = np.unravel_index(cells, (NX, NY))
    t_idx = regime_idx(150, "rich"); T, P = len(t_idx), len(ox)
    for lvl in ("L1", "L3"):
        ctx_g = noise_ctx(ox, oy, t_idx, lvl, "gls"); ctx_w = noise_ctx(ox, oy, t_idx, lvl, "white")
        E_ = np.stack([draw_noise(ctx_g, T, P, s, "h")["h"] for s in range(300)])          # (300, T, P)
        chi_g = np.mean([np.mean(((ctx_g["Ut"].T @ e @ ctx_g["Us"]) * ctx_g["inv"]) ** 2) for e in E_])
        chi_w = np.mean([np.mean((e * ctx_w["inv"]) ** 2) for e in E_])
        var_ratio = float(E_.var() / ctx_g["sig_h_total"] ** 2)
        ok[f"{lvl}_gls_chi2_mean"] = bool(abs(chi_g - 1.0) < 0.02)
        ok[f"{lvl}_white_chi2_mean"] = bool(abs(chi_w - 1.0) < 0.05)
        ok[f"{lvl}_total_variance"] = bool(abs(var_ratio - 1.0) < 0.05)
        print(f"   noise self-test {lvl}: mean whitened chi2 (gls) = {chi_g:.4f}  (white weights) = {chi_w:.4f}  variance / sd_total^2 = {var_ratio:.3f}")
    # sigma_c = 0 must reduce to plain division by sigma_w
    sp = NOISE_LEVELS["L1"]; ctx = noise_ctx(ox, oy, t_idx, "L1", "gls"); r = np.random.default_rng(1).normal(size=(T, P))
    a = (ctx["Ut"].T @ r @ ctx["Us"]) * ctx["inv"]; ok["white_limit_isometry"] = bool(np.allclose(np.sum(a ** 2), np.sum((r / sp["sig_h_white"]) ** 2), rtol=1e-9))
    # empirical spatial correlation of the correlated component vs the kernel (informational, loose tolerance)
    ctx3 = noise_ctx(ox, oy, t_idx, "L3", "gls"); Ks, Kt = _kernels(ox, oy, t_idx)
    Zs = [(ctx3["Ut"] * np.sqrt(ctx3["lt"])) @ np.random.default_rng(s).normal(size=(T, P)) @ (ctx3["Us"] * np.sqrt(ctx3["ls"])).T for s in range(300)]
    emp = np.mean([z.T @ z / T for z in Zs], axis=0); emp_c = emp / np.sqrt(np.outer(np.diag(emp), np.diag(emp)))
    dev = float(np.max(np.abs(emp_c - Ks)))
    ok["spatial_corr_matches_kernel"] = bool(dev < 0.12)
    print(f"   noise self-test: max |empirical - kernel| spatial correlation = {dev:.3f} (tolerance 0.12); sigma_c=0 isometry: {ok['white_limit_isometry']}")
    return bool(all(ok.values())), ok
t0 = time.time(); gate = {}
set_ramp(RAMP_DEFAULT)
if RUN["GATE"]:
    H = run_full_np(CF_T, 1.0, 3000)
    gate["numpy_long_run_finite"] = bool(np.isfinite(H).all() and np.abs(H).max() < 20.0)
    print(f"numpy long run finite and |h|max = {np.abs(H).max():.3f} m  -> {gate['numpy_long_run_finite']}")
    if HAVE_JAX:
        ox, oy = layout_cells(0); n = 300; errs = {}
        for rh in (3.0, 0.0):
            set_ramp(rh); old = CFG["N_STEPS"]; CFG["N_STEPS"] = n
            fwd = make_forward(1.0, ox, oy)
            yj = [np.asarray(a) for a in jax.jit(lambda: fwd(jnp.array(CF_T)))()]
            CFG["N_STEPS"] = old
            yn = run_obs_np(CF_T, 1.0, n, ox, oy)
            errs[rh] = max(float(np.max(np.abs(a - b)) / (np.std(b) + 1e-30)) for a, b in zip(yj, yn))
        set_ramp(RAMP_DEFAULT)
        gate["jax_vs_numpy_forward_max_rel"] = errs
        print("JAX vs NumPy forward model, max |diff|/std (ramp 3 h, ramp 0):", errs)
        nJ = 1500; ox, oy = layout_cells(0)
        bj, Jj = jac_jax(1.0, ox, oy, nJ); bf, Jf = jac_fd(1.0, ox, oy, nJ)
        jerr = [float(np.max(np.abs(a - b)) / (np.max(np.abs(b)) + 1e-30)) for a, b in zip(Jj, Jf)]
        gate["jacobian_jax_vs_fd_rel"] = jerr
        print("Jacobian JAX (jacfwd) vs NumPy central differences, max rel diff for h, u, v:", [f"{x:.2e}" for x in jerr])
        gate["passed"] = bool(gate["numpy_long_run_finite"] and all(e < 1e-8 for e in errs.values()) and all(e < 1e-3 for e in jerr))
    else:
        gate["passed"] = False
        print("JAX unavailable: only the NumPy check ran; E3/E4 will be skipped.")
    ok_n, det_n = noise_selftest()
    gate["noise_selftest"] = ok_n; gate["noise_selftest_detail"] = det_n
    gate["passed"] = bool(gate["passed"] and ok_n)
    RESULTS["gate"] = gate; save_results()
    print("GATE", "PASSED" if gate["passed"] else "FAILED", f"[{time.time()-t0:.0f}s]")
    if HAVE_JAX and not gate["passed"]:
        raise RuntimeError("GATE FAILED - do not interpret any later result; send this output to Claude.")

set_ramp(RAMP_DEFAULT)
assert RAMP_HOURS == RAMP_DEFAULT == 3.0, "ramp must be 3 h after the gate"
print("ramp restored to", RAMP_HOURS, "h")

numpy long run finite and |h|max = 6.782 m  -> True
JAX vs NumPy forward model, max |diff|/std (ramp 3 h, ramp 0): {3.0: 2.067331594027341e-13, 0.0: 1.2618516104024743e-13}
Jacobian JAX (jacfwd) vs NumPy central differences, max rel diff for h, u, v: ['2.00e-07', '7.76e-07', '1.86e-06']
   noise self-test L1: mean whitened chi2 (gls) = 0.9986  (white weights) = 0.9986  variance / sd_total^2 = 0.999
   noise self-test L3: mean whitened chi2 (gls) = 0.9984  (white weights) = 0.9930  variance / sd_total^2 = 0.993
   noise self-test: max |empirical - kernel| spatial correlation = 0.062 (tolerance 0.12); sigma_c=0 isometry: True
GATE PASSED [8s]
ramp restored to 3.0 h


## GATE (stage E only): independent data generator, judged against the smallest noise sd

In [5]:
# ===================== GATE (independent data generator) — runs only when stage E (or ALL) is requested =====================
if RUN["E6"]:
    gg = {}; _rms = lambda a: float(np.sqrt(np.mean(np.asarray(a) ** 2)))
    set_ramp(RAMP_DEFAULT); set_window(150); rich = REGIMES["rich"]
    t0 = time.time()
    # (a) exact volume balance: V(t) = H0*LY*U0*int_0^t ramp(s) cos(w s) ds (RK4 conserves this linear invariant to round-off)
    U0g = CFG["U0_E"][-1]
    _, _, _, vol = run_generator(TRUTH_FN["standard"], U0g, GEN_R, rec_volume=True)
    sfine = np.linspace(0.0, CFG["N_STEPS"] * DT, CFG["N_STEPS"] * 200 + 1)
    integrand = U0g * np.array([float(_ramp(np, s)) for s in sfine]) * np.cos(OMEGA * sfine)
    cum = np.concatenate([[0.0], np.cumsum(0.5 * (integrand[1:] + integrand[:-1]) * np.diff(sfine))])
    vexp = H0 * LY * cum[200::200][:CFG["N_STEPS"]]
    verr = np.max(np.abs(vol - vexp)) / np.max(np.abs(vexp))
    gg["generator_volume"] = bool(verr < 1e-6)
    print(f"(a) generator volume balance: max rel err = {verr:.2e} ->", "PASS" if gg["generator_volume"] else "FAIL")
    # (b) generator convergence: r in {1, GEN_R, 2*GEN_R}; reference = 2*GEN_R
    sig_min = [NOISE_LEVELS["L0"]["sig_h_white"] if NOISE_LEVELS["L0"]["sig_h_corr"] == 0 else math.hypot(NOISE_LEVELS["L0"]["sig_h_white"], NOISE_LEVELS["L0"]["sig_h_corr"]),
               NOISE_LEVELS["L0"]["sig_vel"], NOISE_LEVELS["L0"]["sig_vel"]]
    print(f"    convergence is judged against the SMALLEST noise sd of the notebook (level L0): h {sig_min[0]*100:.1f} cm, u/v {sig_min[1]*100:.2f} cm/s")
    for U0 in CFG["U0_E"]:
        F = {r: run_generator(TRUTH_FN["standard"], U0, r) for r in sorted({1, GEN_R, 2 * GEN_R})}
        ref = F[2 * GEN_R]
        for r in F:
            if r == 2 * GEN_R: continue
            e = [_rms(F[r][k][rich] - ref[k][rich]) / sig_min[k] for k in range(3)]
            tag = ""
            if r == GEN_R:
                okc = bool(max(e) <= 0.25); gg[f"generator_conv_U0={U0}"] = okc; tag = " <- GEN_R: " + ("PASS" if okc else "FAIL (use a larger GEN_R)")
            print(f"(b) U0={U0:<5} r={r}: rms(G_r - G_{2*GEN_R}) / sd_L0   h={e[0]:.3f}  u={e[1]:.3f}  v={e[2]:.3f}{tag}")
    # (c) size of the 'inverse crime' (reported, no pass/fail)
    ax_, ay_ = np.unravel_index(np.arange(NX * NY), (NX, NY)); Xc, Yc = np.meshgrid(XC, YC, indexing="ij"); Tc = TRUTH_FN_SHARP["standard"](Xc, Yc)
    spb = noise_spec(NOISE_LEVEL); sig_base = [math.hypot(spb["sig_h_white"], spb["sig_h_corr"]), spb["sig_vel"], spb["sig_vel"]]
    print(f"(c) model error = rms(inversion solver - generator): absolute, and relative to the BASELINE noise sd ({NOISE_LEVEL}: h {sig_base[0]*100:.1f} cm, u/v {sig_base[1]*100:.1f} cm/s)")
    for U0 in CFG["U0_E"]:
        Hc, Uc, Vc = [a.reshape(CFG["N_STEPS"], NX, NY) for a in run_obs_np(Tc, U0, CFG["N_STEPS"], ax_, ay_)]
        Gf = gen_fields("standard", U0)
        ea = [_rms(a[rich] - b[rich]) for a, b in zip((Hc, Uc, Vc), Gf)]
        print(f"      U0={U0:<5} rich: h={ea[0]*100:.3f} cm ({ea[0]/sig_base[0]:.3f} sd)  u={ea[1]*100:.3f} cm/s ({ea[1]/sig_base[1]:.3f} sd)  v={ea[2]*100:.3f} cm/s ({ea[2]/sig_base[2]:.3f} sd)")
        gg[f"model_error_U0={U0}"] = dict(h_m=ea[0], u_ms=ea[1], v_ms=ea[2], h_rel=ea[0]/sig_base[0], u_rel=ea[1]/sig_base[1], v_rel=ea[2]/sig_base[2])
    gg["generator_ok"] = bool(all(v for k, v in gg.items() if isinstance(v, bool)))
    RESULTS["gate"].update(gg)
    if HAVE_JAX: RESULTS["gate"]["passed"] = bool(RESULTS["gate"].get("passed") and gg["generator_ok"])
    save_results()
    print("GENERATOR GATE", "PASSED" if gg["generator_ok"] else "FAILED", f"[{time.time()-t0:.0f}s]")
    if not gg["generator_ok"]:
        raise RuntimeError("GENERATOR GATE FAILED - do not interpret any later result of stage E; send this output to Claude.")

## Noise calibration: absolute noise expressed as a fraction of the clean signal

In [6]:
# ===================== NOISE CALIBRATION TABLE (NumPy): what the absolute noise means relative to the clean signal =====================
# The earlier notebook used noise = 3% of the window std. Here the noise is absolute; this table shows the EQUIVALENT relative level, so the two notebooks can be compared.
if RUN["GATE"]:
    ox, oy = layout_cells(0); cal_rows = []
    print("clean-signal sd in the early window (layout 0) and the noise sd of every level expressed as a fraction of it")
    print("   U0    start   sd_h[m]  sd_u[m/s] sd_v[m/s] |  " + "  ".join(f"{l}: h/u/v [%]" for l in NOISE_LEVELS))
    for U0 in CFG["U0_E1"]:
        for rh, name in ((3.0, "ramp3"), (0.0, "cold")):
            set_ramp(rh); set_window(150)
            Hc, Uc, Vc = run_obs_np(CF_T, U0, CFG["N_STEPS"], ox, oy); t = REGIMES["rich"]
            s = [float(Hc[t].std()), float(Uc[t].std()), float(Vc[t].std())]
            rel = {l: [100 * math.hypot(sp["sig_h_white"], sp["sig_h_corr"]) / s[0], 100 * sp["sig_vel"] / s[1], 100 * sp["sig_vel"] / s[2]] for l, sp in NOISE_LEVELS.items()}
            cal_rows.append(dict(U0=U0, start=name, sd_h=s[0], sd_u=s[1], sd_v=s[2], rel=rel))
            print(f"   {U0:<5} {name:6s} {s[0]:8.3f} {s[1]:9.4f} {s[2]:9.4f} |  " + "  ".join(f"{l}: {rel[l][0]:5.1f}/{rel[l][1]:5.1f}/{rel[l][2]:6.1f}" for l in NOISE_LEVELS))
    set_ramp(RAMP_DEFAULT); set_window(150)
    RESULTS["noise_cal"] = cal_rows; save_results()
    print("reading: the 3% rule of the earlier notebook corresponds to a 3% column; sd_v is tiny (cross-channel velocity), so any absolute velocity noise of cm/s size is a LARGE fraction of v.")

clean-signal sd in the early window (layout 0) and the noise sd of every level expressed as a fraction of it
   U0    start   sd_h[m]  sd_u[m/s] sd_v[m/s] |  L0: h/u/v [%]  L1: h/u/v [%]  L2: h/u/v [%]  L3: h/u/v [%]
   0.15  ramp3     0.417    0.0621    0.0016 |  L0:   2.4/  8.1/ 309.8  L1:   4.8/ 16.1/ 619.5  L2:  14.0/ 48.3/1858.6  L3:  27.9/ 96.7/3717.2
   0.15  cold      0.441    0.0799    0.0062 |  L0:   2.3/  6.3/  80.1  L1:   4.5/ 12.5/ 160.3  L2:  13.2/ 37.5/ 480.9  L3:  26.4/ 75.1/ 961.8
   0.3   ramp3     0.834    0.1234    0.0031 |  L0:   1.2/  4.1/ 159.4  L1:   2.4/  8.1/ 318.8  L2:   7.0/ 24.3/ 956.4  L3:  14.0/ 48.6/1912.7
   0.3   cold      0.874    0.1533    0.0113 |  L0:   1.1/  3.3/  44.1  L1:   2.3/  6.5/  88.2  L2:   6.7/ 19.6/ 264.7  L3:  13.3/ 39.1/ 529.3
   0.5   ramp3     1.389    0.2042    0.0051 |  L0:   0.7/  2.4/  98.2  L1:   1.4/  4.9/ 196.4  L2:   4.2/ 14.7/ 589.3  L3:   8.4/ 29.4/1178.5
   0.5   cold      1.441    0.2442    0.0168 |  L0:   0.7/  2.0/  29

## E1 — how fast does the start-up transient die?

In [7]:
# ===================== E1: how fast does the start-up transient die? (NumPy only) =====================
if RUN["E1"]:
    set_ramp(RAMP_DEFAULT)
    n = int(CFG["N_LONG_H"] * 3600 / DT)
    c_seiche = math.sqrt(G * H0)
    print(f"reference scales: M2 period P = {2*math.pi/OMEGA/3600:.2f} h | free gravity-wave (seiche) periods 2L/(n c): n=1 {2*LX/c_seiche/3600:.2f} h, n=2 {LX/c_seiche/3600:.2f} h | steady threshold d < {CFG['SS_THRESH']}")
    for U0 in CFG["U0_E1"]:
        key = str(U0)
        if key not in RESULTS["E1"]:
            t0 = time.time()
            Hf = run_full_np(CF_T, U0, n)
            bl = periodicity_defect(Hf); tss = steady_time(bl, CFG["SS_THRESH"]); per, rrel = residual_period(Hf)
            tau_est = 1.0 / (8 / (3 * math.pi) * 2.75e-3 * U0 / H0) / 3600.0
            RESULTS["E1"][key] = dict(blocks=bl, t_ss_h=tss, resid_period_h=per, resid_rel=rrel, tau_est_h=tau_est, run_s=time.time() - t0)
            save_results()
        r = RESULTS["E1"][key]
        d25 = min(r["blocks"], key=lambda b: abs(b[0] - 25.0))[1]
        print(f"U0={U0:5.2f}  defect at ~25 h = {d25:.3f}  t_ss = {r['t_ss_h'] if r['t_ss_h'] is None else round(r['t_ss_h'],1)} h  "
              f"(order-of-magnitude damping time ~{r['tau_est_h']:.0f} h)  leftover oscillation: period {r['resid_period_h']:.2f} h, size {100*r['resid_rel']:.1f}% of the tide signal")
    print("blocks (end time h : defect):")
    for U0 in CFG["U0_E1"]:
        bl = RESULTS["E1"][str(U0)]["blocks"]
        print(f"  U0={U0:5.2f}: " + "  ".join(f"{b[0]:.0f}h:{b[1]:.3f}" for b in bl[::2]))

reference scales: M2 period P = 12.42 h | free gravity-wave (seiche) periods 2L/(n c): n=1 2.97 h, n=2 1.49 h | steady threshold d < 0.05
U0= 0.15  defect at ~25 h = 0.158  t_ss = 124.2 h  (order-of-magnitude damping time ~63 h)  leftover oscillation: period 1.49 h, size 1.0% of the tide signal
U0= 0.30  defect at ~25 h = 0.156  t_ss = 74.5 h  (order-of-magnitude damping time ~32 h)  leftover oscillation: period 1.49 h, size 0.9% of the tide signal
U0= 0.50  defect at ~25 h = 0.153  t_ss = 49.7 h  (order-of-magnitude damping time ~19 h)  leftover oscillation: period 1.49 h, size 0.8% of the tide signal
U0= 1.00  defect at ~25 h = 0.150  t_ss = 37.3 h  (order-of-magnitude damping time ~10 h)  leftover oscillation: period 1.49 h, size 0.5% of the tide signal
blocks (end time h : defect):
  U0= 0.15: 25h:0.158  50h:0.077  75h:0.067  99h:0.057  124h:0.049  149h:0.044  174h:0.037  199h:0.032  224h:0.028
  U0= 0.30: 25h:0.156  50h:0.063  75h:0.048  99h:0.035  124h:0.026  149h:0.020  174h:0.0

## E2 — linearised information vs data window, start type and noise level

In [8]:
# ===================== E2: linearised information vs data window and start type =====================
if RUN["E2"]:
    E2 = RESULTS["E2"]; t0 = time.time()
    cases = [("ramp3", 3.0, CFG["W0_E2_RAMP_H"]), ("cold", 0.0, CFG["W0_E2_COLD_H"])]
    jac = jac_jax if HAVE_JAX else jac_fd
    print("Jacobian engine:", "JAX jacfwd" if HAVE_JAX else "NumPy finite differences (JAX unavailable)")
    for tag, rh, W0h in cases:
        set_ramp(rh)
        for U0 in CFG["U0_E2"]:
            key = f"{tag}|U0={U0}"
            if key in E2: continue
            W0s = [int(round(h * 3600 / DT)) for h in W0h]; n = max(W0s) + CFG["WIN_LEN"]; out = {}
            for layout in range(CFG["N_LAYOUTS"]):
                ox, oy = layout_cells(layout)
                base, J = jac(U0, ox, oy, n)
                for W0, h in zip(W0s, W0h):
                    for variant in ("h", "huv"):
                        for regime in ("rich", "sparse4"):
                            for lvl in CFG["E2_LEVELS"]:
                                out.setdefault(f"W0={h}h|{variant}|{regime}|{lvl}", []).append(fisher_metrics(J, regime_idx(W0, regime), variant, ox, oy, lvl))
            E2[key] = out; save_results()
            print(f"  done {key}  [{time.time()-t0:.0f}s]")
    set_ramp(RAMP_DEFAULT)

## E3 — identification rows: early vs late data window

In [9]:
# ===================== E3: classical identification rows, early vs late window =====================
E3 = RESULTS["E3"]
if RUN["E3"] and HAVE_JAX and RESULTS["gate"].get("passed"):
    set_ramp(RAMP_DEFAULT)
    for U0 in CFG["U0_E3"]:
        W0L, flag = late_W0(U0)
        E3.setdefault("_windows", {})[str(U0)] = dict(W0_late=int(W0L), late_start_h=W0L * DT / 3600.0, flag=flag)
        print(f"U0={U0}: late window starts at step {W0L} ({W0L*DT/3600:.1f} h)  [{flag}]")
        for variant in ("h", "huv"):
            run_row(E3, f"U0={U0}|{variant}|early", U0, variant, CFG["N_SEEDS"], 150)
            run_row(E3, f"U0={U0}|{variant}|late",  U0, variant, CFG["N_SEEDS"], W0L)
    save_results()
    print("(no reproduction check: the noise model differs from the earlier notebook, so its counts are not comparable)")
else:
    print("E3 skipped (RUN flag off, JAX unavailable, or gate not passed).")

E3 skipped (RUN flag off, JAX unavailable, or gate not passed).


## E4 — smooth-truth control

In [10]:
# ===================== E4: smooth truth (model family) control =====================
E4 = RESULTS["E4"]
if RUN["E4"] and HAVE_JAX and RESULTS["gate"].get("passed"):
    set_ramp(RAMP_DEFAULT)
    cf_smooth = np.array(cf_zonal(np, P_TRUE))
    U0 = 1.0
    if U0 in CFG["U0_E3"]:
        W0L = RESULTS["E3"].get("_windows", {}).get(str(U0), {}).get("W0_late") or late_W0(U0)[0]
        for variant in ("h", "huv"):
            run_row(E4, f"U0={U0}|{variant}|early|smooth", U0, variant, CFG["N_SEEDS"], 150, cf_truth=cf_smooth)
            run_row(E4, f"U0={U0}|{variant}|late|smooth",  U0, variant, CFG["N_SEEDS"], W0L, cf_truth=cf_smooth)
        save_results()
else:
    print("E4 skipped (RUN flag off, JAX unavailable, or gate not passed).")

[row] U0=1.0|h|early|smooth                          W0=   2.5 h  noise=L2  identified  1/10  CP95=[0.00,0.45]  -> NOT_IDENTIFIED  chi2_ref(mean)= 1.00  flagged=0  [565s]
[row] U0=1.0|h|late|smooth                           W0=  37.5 h  noise=L2  identified  1/10  CP95=[0.00,0.45]  -> NOT_IDENTIFIED  chi2_ref(mean)= 1.00  flagged=0  [1206s]
[row] U0=1.0|huv|early|smooth                        W0=   2.5 h  noise=L2  identified  5/10  CP95=[0.19,0.81]  -> MARGINAL        chi2_ref(mean)= 1.00  flagged=0  [480s]
[row] U0=1.0|huv|late|smooth                         W0=  37.5 h  noise=L2  identified  2/10  CP95=[0.03,0.56]  -> MARGINAL        chi2_ref(mean)= 1.00  flagged=0  [982s]


## E5 — cold start vs smooth start at the optimiser level (own pre-registration)

In [11]:
# ===================== E5 (stage D): close the transient loop at the optimiser level =====================
def defect_at(U0, ramp_h, t_h):
    """periodicity defect (NumPy) of the block ending closest to t_h hours, for the given ramp."""
    set_ramp(ramp_h); n = int((t_h + 30) * 3600 / DT)
    bl = periodicity_defect(run_full_np(CF_T, U0, n)); set_ramp(RAMP_DEFAULT)
    b = min(bl, key=lambda x: abs(x[0] - t_h)); return b

E5 = RESULTS["E5"]
if RUN["E5"]:
    print("stage-D pre-registration hash:", PREREG_D_HASH)
    W0D = int(round(CFG["W0_D_LATE_H"] * 3600 / DT))
    for ramp_h, name in ((3.0, "ramp 3 h"), (0.0, "cold start")):
        d25 = defect_at(1.0, ramp_h, 25.0); dl = defect_at(1.0, ramp_h, CFG["W0_D_LATE_H"])
        print(f"U0=1, {name:10s}: periodicity defect ~{d25[0]:.0f} h = {d25[1]:.3f} ; ~{dl[0]:.0f} h = {dl[1]:.4f}")
if RUN["E5"] and HAVE_JAX and RESULTS["gate"].get("passed"):
    for U0 in CFG["U0_D"]:
        set_ramp(3.0); run_row(E5, f"ramp3|U0={U0}|h|early", U0, "h", CFG["N_SEEDS"], 150)
        set_ramp(0.0); run_row(E5, f"cold|U0={U0}|h|early",  U0, "h", CFG["N_SEEDS"], 150)
    if 1.0 in CFG["U0_D"]:
        set_ramp(3.0); run_row(E5, "ramp3|U0=1.0|h|late", 1.0, "h", CFG["N_SEEDS"], W0D)
        set_ramp(0.0); run_row(E5, "cold|U0=1.0|h|late",  1.0, "h", CFG["N_SEEDS"], W0D)
    set_ramp(RAMP_DEFAULT); save_results()
else:
    print("E5 skipped (RUN flag off, JAX unavailable, or gate not passed).")

E5 skipped (RUN flag off, JAX unavailable, or gate not passed).


## E6 — independent data solver vs inversion solver (own pre-registration)

In [12]:
# ===================== stage E: independent data solver ('indep' arm) vs data from the inversion solver ('same' arm) =====================
E6 = RESULTS["E6"]
if RUN["E6"] and HAVE_JAX and RESULTS["gate"].get("passed"):
    print("stage-E pre-registration hash:", PREREG_E_HASH)
    set_ramp(RAMP_DEFAULT)
    for U0 in CFG["U0_E"]:
        for variant in ("h", "huv"):
            run_row(E6, f"same|U0={U0}|{variant}|early",  U0, variant, CFG["N_SEEDS"], 150, arm="same")
            run_row(E6, f"indep|U0={U0}|{variant}|early", U0, variant, CFG["N_SEEDS"], 150, arm="indep")
    save_results()
    print("(no reproduction check: the noise model differs from the earlier notebook)")
else:
    print("stage E skipped (RUN flag off, JAX unavailable, or gate not passed).")

stage E skipped (RUN flag off, JAX unavailable, or gate not passed).


## E7 — noise sweep (stage F, own pre-registration)

In [13]:
# ===================== stage F: noise sweep (identification rows at other noise levels) =====================
E7 = RESULTS["E7"]
if RUN["E7"] and HAVE_JAX and RESULTS["gate"].get("passed"):
    print("stage-F pre-registration hash:", PREREG_F_HASH, "| levels swept:", CFG["F_LEVELS"], "| baseline level (stage B rows):", NOISE_LEVEL)
    set_ramp(RAMP_DEFAULT)
    for U0 in CFG["U0_F"]:
        for lvl in CFG["F_LEVELS"]:
            if lvl == NOISE_LEVEL: continue
            for variant in ("h", "huv"):
                run_row(E7, f"{lvl}|U0={U0}|{variant}|early", U0, variant, CFG["N_SEEDS"], 150, level=lvl)
    save_results()
else:
    print("stage F skipped (RUN flag off, JAX unavailable, or gate not passed).")

stage F skipped (RUN flag off, JAX unavailable, or gate not passed).


## SUMMARY: tables, pre-registered verdicts, figures

In [14]:
# ===================== SUMMARY: tables, pre-registered verdicts, figures =====================
# fill sections missing in this session from the archive copies of other sessions (same MODE, baseline level and pre-registration hash)
for _p in sorted(glob.glob(os.path.join(OUT_DIR, f"twin2_results_{MODE}_{NOISE_LEVEL}_stage*.json")) + glob.glob(f"/kaggle/input/**/twin2_results_{MODE}_{NOISE_LEVEL}_stage*.json", recursive=True)):
    try:
        _o = json.load(open(_p))
        if _o.get("prereg_hash") != PREREG_HASH or _o.get("mode") != MODE: continue
        for _k in ("noise_cal", "E1", "E2", "E3", "E4", "E5", "E6", "E7"):
            if not RESULTS[_k] and _o.get(_k): RESULTS[_k] = _o[_k]; print("filled", _k, "from", os.path.basename(_p))
    except Exception as _e: print("could not read", _p, _e)

def med(entries, field): return float(np.median([e[field] for e in entries]))
print("=" * 110); print("Twin experiment | prereg hashes", PREREG_HASH, PREREG_D_HASH, PREREG_E_HASH, PREREG_F_HASH, "| mode", MODE, "| baseline noise", NOISE_LEVEL, "| weighting", WEIGHTING, "| JAX", HAVE_JAX); print("=" * 110)
print("noise levels:"); [print(f"   {k}: {v['desc']}") for k, v in NOISE_LEVELS.items()]

# ---- noise calibration ----
if RESULTS["noise_cal"]:
    print("\nNOISE CALIBRATION  (equivalent relative noise = absolute sd / clean-signal sd, early window; %)")
    for r in RESULTS["noise_cal"]:
        print(f"   U0={r['U0']:<5} {r['start']:6s} sd_h={r['sd_h']:.3f} m  " + "  ".join(f"{l}:h {r['rel'][l][0]:.1f}%/u {r['rel'][l][1]:.1f}%/v {r['rel'][l][2]:.0f}%" for l in NOISE_LEVELS))

# ---- E1 ----
print("\nE1  transient decay (truth Cf, ramp 3 h; independent of the noise model)")
for U0 in CFG["U0_E1"]:
    r = RESULTS["E1"].get(str(U0))
    if r: print(f"   U0={U0:5.2f}  t_ss = {r['t_ss_h'] if r['t_ss_h'] is None else round(r['t_ss_h'],1)} h   estimate ~{r['tau_est_h']:.0f} h   leftover period {r['resid_period_h']:.2f} h")

# ---- E2 ----
print("\nE2  linearised uncertainty (median over layouts, exact space-time noise covariance).  sd_bnd = rms boundary-position error [km].  ratio = sd_bnd / sd_bnd(ramp 3 h, window 2.5 h, same level)")
def _w(k): return float(k.split("|")[0][3:-1])
verdict = {}
for U0 in CFG["U0_E2"]:
    print(f"\n   U0 = {U0}")
    for lvl in CFG["E2_LEVELS"]:
        ref = RESULTS["E2"].get(f"ramp3|U0={U0}", {}).get(f"W0=2.5h|h|rich|{lvl}")
        if not ref: continue
        ref_b = med(ref, "sd_bnd_km")
        print(f"   --- level {lvl}" + ("  (baseline)" if lvl == NOISE_LEVEL else ""))
        print("   start    window   variant regime  sd_lnClow sd_lnChigh sd_bnd_km  ratio")
        for tag in ("cold", "ramp3"):
            E = RESULTS["E2"].get(f"{tag}|U0={U0}", {})
            for k in sorted([k for k in E if k.endswith("|" + lvl)], key=lambda s: (_w(s), s)):
                w, var, reg, _ = k.split("|")
                if reg == "sparse4" and var == "huv": continue
                m = E[k]
                print(f"   {tag:6s}  {w:9s} {var:6s} {reg:8s} {med(m,'sd_lnClow'):9.3f} {med(m,'sd_lnChigh'):10.3f} {med(m,'sd_bnd_km'):9.2f} {med(m,'sd_bnd_km')/ref_b:7.2f}")
        cold = RESULTS["E2"].get(f"cold|U0={U0}", {}).get(f"W0=2.5h|h|rich|{lvl}")
        late = [med(v, "sd_bnd_km") for k, v in RESULTS["E2"].get(f"ramp3|U0={U0}", {}).items() if k.endswith(f"|h|rich|{lvl}") and _w(k) >= 24.0]
        verdict.setdefault(lvl, {})[U0] = dict(A=(med(cold, "sd_bnd_km") / ref_b) if cold else None, B=(min(late) / ref_b) if late else None)
print("\nPRE-REGISTERED VERDICTS (E2) at the baseline level", NOISE_LEVEL)
vb = verdict.get(NOISE_LEVEL, {})
A_list = [(u, v["A"]) for u, v in vb.items() if u >= 0.5 and v["A"] is not None]; B_list = [(u, v["B"]) for u, v in vb.items() if v["B"] is not None]
if A_list: print("   TH-A (cold start >= 2x more informative, U0>=0.5):", "SUPPORTED" if all(x <= 0.5 for _, x in A_list) else "NOT SUPPORTED", {u: round(x, 2) for u, x in A_list})
else: print("   TH-A: not evaluable")
if B_list: print("   TH-B (no later window >= 1.5x better):", "SUPPORTED" if all(x >= 0.67 for _, x in B_list) else "NOT SUPPORTED", {u: round(x, 2) for u, x in B_list})
else: print("   TH-B: not evaluable")
print("   informational, other levels: " + "; ".join(f"{l}: A={ {u: round(v['A'],2) for u, v in d.items() if u >= 0.5 and v['A'] is not None} } B={ {u: round(v['B'],2) for u, v in d.items() if v['B'] is not None} }" for l, d in verdict.items() if l != NOISE_LEVEL))
RESULTS["verdict_E2"] = {l: dict(A={str(u): v["A"] for u, v in d.items()}, B={str(u): v["B"] for u, v in d.items()}) for l, d in verdict.items()}

# ---- rows ----
def _ci(row):
    c = classify_row(row["seeds"]); return c["identified"], c["n_seeds"], cp_interval(c["identified"], c["n_seeds"]), c["cls"]
def print_rows(title, store):
    print(f"\n{title}")
    for tag, row in store.items():
        if tag.startswith("_"): continue
        k, n, (lo, hi), cls = _ci(row); cr = [s["chi2_ref"] for s in row["seeds"]]
        print(f"   {tag:36s} W0={row['W0']*DT/3600:6.1f} h  noise={row.get('level','?')}  identified {k:2d}/{n}  CP95=[{lo:.2f},{hi:.2f}]  {cls:15s} chi2_ref mean/median/max={np.mean(cr):.2f}/{np.median(cr):.2f}/{np.max(cr):.2f}")
def _disjoint(a, b): return a[2][0] > b[2][1] or b[2][0] > a[2][1]
if RESULTS["E3"]: print_rows(f"E3  classical rows, sharp truth, baseline noise {NOISE_LEVEL}", RESULTS["E3"])
if RESULTS["E4"]: print_rows("E4  classical rows, smooth truth (model family)", RESULTS["E4"])
if RESULTS["E3"]:
    print("\nE3 pre-registered reading: 'late better than early' only if the 95% intervals do not overlap")
    for U0 in CFG["U0_E3"]:
        for var in ("h", "huv"):
            a, b = RESULTS["E3"].get(f"U0={U0}|{var}|early"), RESULTS["E3"].get(f"U0={U0}|{var}|late")
            if a and b:
                ca, cb = _ci(a), _ci(b); res = "late better" if cb[2][0] > ca[2][1] else ("early better" if ca[2][0] > cb[2][1] else "not distinguishable")
                print(f"   U0={U0} {var:3s}: early {ca[0]}/{ca[1]}  late {cb[0]}/{cb[1]}  -> {res}")
if RESULTS["E5"]:
    print_rows("E5  stage D: cold start vs ramp, early vs late (h-only, sharp truth)", RESULTS["E5"])
    g = lambda t: (_ci(RESULTS["E5"][t]) if t in RESULTS["E5"] else None)
    ce, re_, cl, rl = g("cold|U0=1.0|h|early"), g("ramp3|U0=1.0|h|early"), g("cold|U0=1.0|h|late"), g("ramp3|U0=1.0|h|late")
    print("\nPRE-REGISTERED VERDICTS (stage D)")
    if ce and re_:
        D1 = ce[2][0] > re_[2][1]
        print(f"   D1 (cold early {ce[0]}/{ce[1]} vs ramp early {re_[0]}/{re_[1]}, intervals non-overlapping):", "SUPPORTED" if D1 else "NOT SUPPORTED")
        if cl and rl: print(f"   D2 (D1 and cold late {cl[0]}/{cl[1]} not distinguishable from ramp late {rl[0]}/{rl[1]}):", "SUPPORTED" if (D1 and not _disjoint(cl, rl)) else "NOT SUPPORTED")
    if re_ and rl: print(f"   D3 (late ramp-3 window helps: late {rl[0]}/{rl[1]} vs early {re_[0]}/{re_[1]}):", "SUPPORTED" if rl[2][0] > re_[2][1] else "NOT SUPPORTED")
if RESULTS["E6"]:
    print_rows("E6  stage E: data from the inversion solver (same) vs the independent generator (indep)", RESULTS["E6"])
    print("\nPRE-REGISTERED READING (stage E): 'independent generator changes identification' only if the 95% intervals do not overlap")
    for U0 in CFG["U0_E"]:
        for var in ("h", "huv"):
            a, b = RESULTS["E6"].get(f"same|U0={U0}|{var}|early"), RESULTS["E6"].get(f"indep|U0={U0}|{var}|early")
            if a and b:
                ca, cb = _ci(a), _ci(b); print(f"   U0={U0} {var:3s}: same {ca[0]}/{ca[1]}  indep {cb[0]}/{cb[1]}  -> {'changes' if _disjoint(ca, cb) else 'no detectable change'}")
if RESULTS["E7"]:
    print_rows("E7  stage F: noise sweep (ramp 3 h, early window, sharp truth)", RESULTS["E7"])
    print("\nPRE-REGISTERED READINGS (stage F)")
    lvls = sorted(CFG["F_LEVELS"], key=lambda l: math.hypot(NOISE_LEVELS[l]["sig_h_white"], NOISE_LEVELS[l]["sig_h_corr"]))
    lowest = lvls[0]; f1 = {}
    for U0 in CFG["U0_F"]:
        r = RESULTS["E7"].get(f"{lowest}|U0={U0}|h|early")
        if r: f1[U0] = classify_row(r["seeds"])["cls"]
    if f1:
        if all(c == "NOT_IDENTIFIED" for c in f1.values()): txt = "STRUCTURAL: h-only non-identification persists at the lowest noise level tested (not noise-limited within the range)"
        elif any(c == "IDENTIFIED" for c in f1.values()): txt = "NOISE-LIMITED for at least one U0 (identified at the lowest level); see the table for sigma*"
        else: txt = "INTERMEDIATE"
        print(f"   F1 (lowest level {lowest}: {f1}):", txt)
    base_rows = RESULTS["E3"]; f2 = []
    for U0 in CFG["U0_F"]:
        a, b = base_rows.get(f"U0={U0}|h|early"), base_rows.get(f"U0={U0}|huv|early")
        if a and b:
            ca, cb = _ci(a), _ci(b); f2.append((U0, ca[0], cb[0], cb[2][0] > ca[2][1]))
    if f2: print(f"   F2 (adding velocity helps at baseline {NOISE_LEVEL}; (U0, h identified, huv identified, lower bound huv > upper bound h)):", "SUPPORTED" if any(x[3] for x in f2) else "NOT SUPPORTED", f2)
    else: print("   F2: not evaluable (stage-B rows at the baseline level are missing)")
    print("   F3 (monotonic degradation, reported not tested): identified counts by level")
    for U0 in CFG["U0_F"]:
        for var in ("h", "huv"):
            cnt = {}
            for l in lvls:
                r = RESULTS["E7"].get(f"{l}|U0={U0}|{var}|early")
                if r: cnt[l] = classify_row(r["seeds"])["identified"]
            b = base_rows.get(f"U0={U0}|{var}|early")
            if b: cnt[NOISE_LEVEL] = classify_row(b["seeds"])["identified"]
            order = sorted(cnt, key=lambda l: math.hypot(NOISE_LEVELS[l]["sig_h_white"], NOISE_LEVELS[l]["sig_h_corr"]))
            seq = [cnt[l] for l in order]
            print(f"      U0={U0} {var:3s}: " + "  ".join(f"{l}:{cnt[l]}" for l in order) + "   non-increasing: " + str(all(seq[i] >= seq[i + 1] for i in range(len(seq) - 1))))
save_results()
_snap = os.path.join(OUT_DIR, f"twin2_results_{SNAP_TAG}.json")
with open(_snap, "w") as f: json.dump(RESULTS, f, default=_jsonable)
print("\narchive copy of this session:", _snap, "| cumulative file:", OUT_PATH)

# ---- figures ----
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    for U0 in CFG["U0_E1"]:
        r = RESULTS["E1"].get(str(U0))
        if r: ax[0].semilogy([b[0] for b in r["blocks"]], [max(b[1], 1e-5) for b in r["blocks"]], "o-", label=f"U0={U0}")
    ax[0].axhline(CFG["SS_THRESH"], color="k", ls="--", lw=0.8); ax[0].axvspan(2.5, 25, color="orange", alpha=0.15, label="early data window")
    ax[0].set_xlabel("time [h]"); ax[0].set_ylabel("periodicity defect of h"); ax[0].set_title("E1: how fast the transient dies"); ax[0].legend(fontsize=8)
    for tag, ls in (("cold", "--"), ("ramp3", "-")):
        for U0 in CFG["U0_E2"]:
            E = RESULTS["E2"].get(f"{tag}|U0={U0}", {})
            ks = sorted([k for k in E if k.endswith(f"|h|rich|{NOISE_LEVEL}")], key=_w)
            if ks: ax[1].semilogy([_w(k) for k in ks], [med(E[k], "sd_bnd_km") for k in ks], "o" + ls, label=f"{tag} U0={U0}")
    ax[1].set_xlabel("window start [h]"); ax[1].set_ylabel(f"sd of boundary position [km] (h-only, rich, {NOISE_LEVEL})"); ax[1].set_title("E2: linearised information"); ax[1].legend(fontsize=7)
    plt.tight_layout(); _fp = os.path.join(OUT_DIR, f"twin2_summary_{SNAP_TAG}.png"); plt.savefig(_fp, dpi=130); print("\nfigure saved:", _fp)
except Exception as e: print("figure skipped:", e)

try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    groups = [("E3", RESULTS["E3"]), ("E4", RESULTS["E4"]), ("E5", RESULTS["E5"]), ("E6", RESULTS["E6"]), ("E7", RESULTS["E7"])]
    items = [(g + " " + tag, row) for g, st in groups for tag, row in st.items() if not tag.startswith("_")]
    if items:
        fig, ax = plt.subplots(figsize=(11, 0.38 * len(items) + 1.5))
        for i, (name, row) in enumerate(items):
            k, n, (lo, hi), _ = _ci(row); p = k / n
            ax.errorbar(p, i, xerr=[[p - lo], [hi - p]], fmt="o", capsize=3, color=("tab:green" if "|huv|" in name else "tab:blue"))
        ax.set_yticks(range(len(items))); ax.set_yticklabels([n for n, _ in items], fontsize=7); ax.invert_yaxis()
        ax.set_xlim(-0.03, 1.03); ax.axvline(0.9, color="k", ls="--", lw=0.6); ax.axvline(0.1, color="k", ls="--", lw=0.6)
        ax.set_xlabel("fraction of noise realisations identified (95% Clopper-Pearson)   [blue: h-only, green: h+u+v]"); ax.set_title(f"identification rows (baseline noise {NOISE_LEVEL}; E7 rows carry their own level in the tag)"); plt.tight_layout()
        _fp2 = os.path.join(OUT_DIR, f"twin2_rows_{SNAP_TAG}.png"); plt.savefig(_fp2, dpi=130); print("figure saved:", _fp2)
except Exception as e: print("rows figure skipped:", e)

try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    if RESULTS["E7"]:
        fig, ax = plt.subplots(figsize=(7, 4.2)); sig = lambda l: 100 * math.hypot(NOISE_LEVELS[l]["sig_h_white"], NOISE_LEVELS[l]["sig_h_corr"])
        for U0 in CFG["U0_F"]:
            for var, col in (("h", "tab:blue"), ("huv", "tab:green")):
                pts = []
                for l in NOISE_LEVELS:
                    r = RESULTS["E7"].get(f"{l}|U0={U0}|{var}|early") if l != NOISE_LEVEL else RESULTS["E3"].get(f"U0={U0}|{var}|early")
                    if r: k, n, (lo, hi), _ = _ci(r); pts.append((sig(l), k / n, lo, hi))
                if pts:
                    pts.sort(); xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
                    ax.errorbar(xs, ys, yerr=[[p[1] - p[2] for p in pts], [p[3] - p[1] for p in pts]], fmt="o-" if U0 == max(CFG["U0_F"]) else "s--", capsize=3, color=col, alpha=0.9 if U0 == max(CFG["U0_F"]) else 0.55, label=f"{var} U0={U0}")
        ax.set_xscale("log"); ax.set_xlabel("total SLA noise sd [cm] (levels L0..L3)"); ax.set_ylabel("fraction identified (CP95)"); ax.set_ylim(-0.03, 1.03); ax.legend(fontsize=7); ax.set_title("F: identification vs noise level (ramp 3 h, early window)")
        plt.tight_layout(); _fp3 = os.path.join(OUT_DIR, f"twin2_noise_sweep_{SNAP_TAG}.png"); plt.savefig(_fp3, dpi=130); print("figure saved:", _fp3)
except Exception as e: print("noise-sweep figure skipped:", e)

Twin experiment | prereg hashes fc780ac77f513334 086c7a70e942c1e9 609fe5ea67ae1b25 848edd06c215007a | mode standard | baseline noise L2 | weighting gls | JAX True
noise levels:
   L0: diagnostic floor: 1 cm white SLA, 0.5 cm/s velocity (asks whether ANY noise level makes h-only identifiable)
   L1: optimistic: 2 cm white SLA, 1 cm/s velocity
   L2: baseline: 5 cm white + 3 cm correlated SLA (25 km, 3 h), 3 cm/s velocity
   L3: pessimistic: 10 cm white + 6 cm correlated SLA, 6 cm/s velocity

NOISE CALIBRATION  (equivalent relative noise = absolute sd / clean-signal sd, early window; %)
   U0=0.15  ramp3  sd_h=0.417 m  L0:h 2.4%/u 8.1%/v 310%  L1:h 4.8%/u 16.1%/v 620%  L2:h 14.0%/u 48.3%/v 1859%  L3:h 27.9%/u 96.7%/v 3717%
   U0=0.15  cold   sd_h=0.441 m  L0:h 2.3%/u 6.3%/v 80%  L1:h 4.5%/u 12.5%/v 160%  L2:h 13.2%/u 37.5%/v 481%  L3:h 26.4%/u 75.1%/v 962%
   U0=0.3   ramp3  sd_h=0.834 m  L0:h 1.2%/u 4.1%/v 159%  L1:h 2.4%/u 8.1%/v 319%  L2:h 7.0%/u 24.3%/v 956%  L3:h 14.0%/u 48.6%/v 191

/tmp/ipykernel_23/2458408231.py:148: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax[1].set_xlabel("window start [h]"); ax[1].set_ylabel(f"sd of boundary position [km] (h-only, rich, {NOISE_LEVEL})"); ax[1].set_title("E2: linearised information"); ax[1].legend(fontsize=7)



figure saved: /kaggle/working/twin2_summary_standard_L2_stageC.png
figure saved: /kaggle/working/twin2_rows_standard_L2_stageC.png
